# S3 Object Storage on FABRIC

FABRIC's Ceph clusters expose an **S3-compatible object store** (Ceph RGW) alongside
the POSIX CephFS filesystem shown in the other notebooks in this directory.

This notebook covers what FABlib does for S3:

1. Discover which clusters offer S3 and where their endpoints are
2. List the buckets you own
3. Fetch your S3 credentials and write ready-to-use client config
4. Use those credentials from a FABRIC node with `aws-cli`, `s3cmd`, or `boto3`

### Two things worth knowing up front

**FABlib does not move object data.** It is a control-plane helper: it finds
endpoints, lists buckets, and hands you credentials. Transfers are done with a
standard S3 client so that FABlib does not drag a `boto3` dependency into every
FABRIC install. Everything below works with any S3 tool you already use.

**Bucket creation and deletion are administrative.** Only FABRIC facility
administrators and owners of the *Service - FABRIC Ceph* project can create or
delete buckets. Request one through the **Storage → S3 Buckets** tab of the
Credential Manager portal. Once a bucket is yours, you have full read/write
access to its contents.

**Your S3 identity is your bastion login** — the same identity used to name your
CephFS subvolume.

## Setup

In [ ]:
from fabrictestbed_extensions.fablib.fablib import FablibManager as fablib_manager

fablib = fablib_manager()
fablib.show_config();

## 1. Discover clusters and their S3 endpoints

Each FABRIC Ceph cluster is an independent RGW realm, so an S3 user and its keys
exist **per cluster** — the same uid on `east` and `west` are unrelated accounts
with different credentials.

In [ ]:
import json

clusters = fablib.discover_ceph_clusters()
print(f"Available storage clusters: {json.dumps(clusters, indent=2)}")

# Pick one to work with for the rest of the notebook.
CLUSTER = clusters[0] if clusters else "east"
print(f"\nUsing cluster: {CLUSTER}")

In [ ]:
from fabrictestbed_extensions.utils.ceph_s3_utils import CephS3Credentials

endpoints = CephS3Credentials.list_s3_endpoints(
    base_url=fablib.get_ceph_mgr_host(),
    cluster=CLUSTER,
    token_file=fablib.get_token_location(),
)
print(f"S3 endpoints for {CLUSTER}:")
for e in endpoints:
    print(f"  {e}")

print("\nNote: these are FABNet addresses. They are reachable from a FABRIC")
print("slice (any node with a FABNet interface), not from the public internet.")

## 2. List your buckets

This goes through the Ceph Manager API, so it needs no S3 client at all.

The service scopes the result by identity: unless you are an administrator you
will only ever see buckets you own, regardless of what you ask for.

In [ ]:
buckets = fablib.list_s3_buckets(cluster=CLUSTER)

if not buckets:
    print(f"No buckets found on {CLUSTER}.")
    print("Request one via the Credential Manager portal: Storage -> S3 Buckets.")
else:
    for b in buckets:
        print(f"  {b['name']:30} owner={b.get('owner'):20} "
              f"objects={b.get('num_objects')} versioning={b.get('versioning')}")

## 3. Get your S3 credentials

`get_s3_credentials()` returns an access key / secret key pair plus the endpoint
to point a client at.

An existing keypair is reused when one can be read back; a new one is minted only
if you have none. That matters — otherwise every call would leave another key
behind on your account.

> The secret is a long-lived credential. Treat the files below like an SSH private
> key: do not commit them, and do not paste them into a shared notebook output.

In [ ]:
creds = fablib.get_s3_credentials(cluster=CLUSTER)

print(f"cluster    : {creds['cluster']}")
print(f"uid        : {creds['uid']}")
print(f"endpoint   : {creds['endpoint']}")
print(f"access_key : {creds['access_key']}")
print(f"secret_key : {'*' * len(creds['secret_key'])}  ({len(creds['secret_key'])} chars)")

### Write ready-to-use client config

Passing `out_base` also writes config for the common S3 clients. Files holding a
secret are written mode `0600`.

In [ ]:
creds = fablib.get_s3_credentials(cluster=CLUSTER, out_base="./s3-artifacts")

for name, path in sorted(creds["files"].items()):
    print(f"  {name:16} {path}")

print("\n--- README.md ---")
print(open(creds["files"]["README.md"]).read())

## 4. Use the credentials

### From your local machine / JupyterHub

Only works if you have FABNet routing to the endpoint. From a FABRIC node it
always works. `boto3` is not required by FABlib, so install it if you want to
drive S3 from Python:

```bash
pip install boto3
```

In [ ]:
# Uncomment to use boto3 directly (requires `pip install boto3` and FABNet routing).
#
# import boto3
# s3 = boto3.client(
#     "s3",
#     endpoint_url=creds["endpoint"],
#     aws_access_key_id=creds["access_key"],
#     aws_secret_access_key=creds["secret_key"],
#     region_name=creds["region"],
#     # RGW does not serve virtual-host style addressing by default.
#     config=boto3.session.Config(signature_version="s3v4",
#                                 s3={"addressing_style": "path"}),
# )
#
# BUCKET = buckets[0]["name"]          # a bucket you own
# s3.upload_file("./s3-artifacts/README.md", BUCKET, "README.md")
# for obj in s3.list_objects_v2(Bucket=BUCKET).get("Contents", []):
#     print(obj["Key"], obj["Size"])
# s3.download_file(BUCKET, "README.md", "./downloaded-README.md")
print("boto3 snippet above is commented out — uncomment once you own a bucket.")

## 5. Use S3 from a FABRIC node

This is the common case: a slice node reads and writes objects over FABNet.
The node needs a FABNet interface, which `add_fabnet()` provides.

In [ ]:
slice_name = "S3-Example"

slice1 = fablib.new_slice(name=slice_name)
node = slice1.add_node(name="s3-client", cores=2, ram=8, disk=50)
node.add_fabnet()          # required: RGW endpoints live on FABNet

slice1.submit();

In [ ]:
slice1 = fablib.get_slice(slice_name)
node = slice1.get_node(name="s3-client")

# Install a client. awscli is in the distro repos for Rocky/Ubuntu.
node.execute("sudo dnf install -y awscli || sudo apt-get install -y -qq awscli")

### Push the credentials to the node and run a round trip

In [ ]:
BUCKET = buckets[0]["name"] if buckets else "<your-bucket>"

node.execute("mkdir -p ~/.aws")
node.upload_file(creds["files"]["aws_credentials"], ".aws/credentials")
node.upload_file(creds["files"]["aws_config"], ".aws/config")
node.execute("chmod 600 ~/.aws/credentials")

profile = f"fabric-{CLUSTER}"
endpoint = creds["endpoint"]

# Round trip: write a file, upload, list, download, compare.
node.execute(f"""
set -e
echo 'hello from FABRIC' > /tmp/hello.txt
aws --profile {profile} --endpoint-url {endpoint} s3 cp /tmp/hello.txt s3://{BUCKET}/hello.txt
aws --profile {profile} --endpoint-url {endpoint} s3 ls s3://{BUCKET}/
aws --profile {profile} --endpoint-url {endpoint} s3 cp s3://{BUCKET}/hello.txt /tmp/hello-back.txt
diff /tmp/hello.txt /tmp/hello-back.txt && echo 'ROUND TRIP OK'
""")

### s3cmd

`write_client_config()` also emits an `.s3cfg` if you prefer `s3cmd`.

In [ ]:
# node.execute("sudo dnf install -y s3cmd || sudo apt-get install -y -qq s3cmd")
# node.upload_file(creds["files"]["s3cfg"], ".s3cfg")
# node.execute("chmod 600 ~/.s3cfg")
# node.execute(f"s3cmd ls s3://{BUCKET}/")
print("s3cmd snippet above is commented out.")

## Cleanup

In [ ]:
slice1.delete()

## Reference

| What | How |
|---|---|
| Which clusters exist | `fablib.discover_ceph_clusters()` |
| S3 endpoints for a cluster | `CephS3Credentials.list_s3_endpoints(...)` |
| Buckets you own | `fablib.list_s3_buckets(cluster=...)` |
| Credentials + client config | `fablib.get_s3_credentials(cluster=..., out_base=...)` |
| Create / delete a bucket | Credential Manager portal → Storage → S3 Buckets (admins only) |
| Transfer objects | any S3 client, using the config written above |

**POSIX vs S3.** Use CephFS (the other notebooks here) when you want a shared
filesystem your jobs can `open()` and `mmap()`. Use S3 when you want
HTTP-accessible objects, versioning, or data shared across slices without a
mount. Both are backed by the same Ceph clusters.